In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

**<font size="6" color="red">ch3. 연관 분석 </font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고, 그 속성들 사이에 연관성이 어느정도 있는지 분석
- 활용 분야 : 이벤트 사전 감지(사기 적발..), 신상품 카테고리 구성, 상품 진열..
- [조건 : left-hand side:오렌지주스] -> [결과 : right-hand side:와인]
    1. 지지도(support) : 얼마나 자주 함께 등장하는지
        (lhs, rhs)의 항목수 / 전체 항목수 = 1 / 5 = 0.2
    2. 신뢰도(confidence) : 조건이 오면 결과가 얼마나 자주 나타나는지
        (lhs -> rhs)의 항목수 / lhs의 항목수 = 1/2 = 0.5
    3. 향상도(lift) : 우연히 발생한 규칙은 아닌지 확인
        lhs->rhs의 지지도 / (lhs의 지지도 * rhs의 지지도)<br>
        => 0.2 / (0.4 * 0.6) = 0.833
        => 향상도 < 1 : 기대가 낮다 / 향상도 > 1 : 기대가 있다

In [2]:
import csv
# transaction = []
with open('data/cf_basket.csv', 'r', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [9]:
from apyori import apriori
rules = apriori(transaction, # 2차원 데이터
               min_support = 0.15,
               min_confidence=0.1)
rules = list(rules)
len(rules)

18

In [12]:
rules[17]

RelationRecord(items=frozenset({'콜라', '소주', '와인'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'콜라', '소주', '와인'}), confidence=0.2, lift=1.0), OrderedStatistic(items_base=frozenset({'소주'}), items_add=frozenset({'콜라', '와인'}), confidence=0.33333333333333337, lift=0.8333333333333334), OrderedStatistic(items_base=frozenset({'와인'}), items_add=frozenset({'콜라', '소주'}), confidence=0.33333333333333337, lift=0.5555555555555557), OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주', '와인'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'소주', '와인'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25), OrderedStatistic(items_base=frozenset({'콜라', '소주'}), items_add=frozenset({'와인'}), confidence=0.33333333333333337, lift=0.5555555555555557), OrderedStatistic(items_base=frozenset({'콜라', '와인'}), items_add=frozenset({'소주'}), confidence=0.5, lift=0.8333333333333334)])

In [24]:
for rule in rules:
    supprot = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([ data for data in item[0]])
        rhs = ', '.join([ data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift >1:
            print("{} => {}\t {}\t {}\t {}\t".format(lhs, rhs, supprot, 
                                                     round(confidence, 2),
                                                     round(lift,2)))

맥주 => 콜라	 0.4	 1.0	 1.25	
콜라 => 맥주	 0.4	 0.5	 1.25	
소주 => 콜라	 0.6	 1.0	 1.25	
콜라 => 소주	 0.6	 0.75	 1.25	
콜라 => 맥주, 소주	 0.2	 0.25	 1.25	
맥주, 소주 => 콜라	 0.2	 1.0	 1.25	
맥주 => 콜라, 와인	 0.2	 0.5	 1.25	
콜라 => 맥주, 와인	 0.2	 0.25	 1.25	
맥주, 와인 => 콜라	 0.2	 1.0	 1.25	
콜라, 와인 => 맥주	 0.2	 0.5	 1.25	
소주 => 오렌지주스, 콜라	 0.2	 0.33	 1.67	
콜라 => 오렌지주스, 소주	 0.2	 0.25	 1.25	
오렌지주스, 소주 => 콜라	 0.2	 1.0	 1.25	
콜라, 오렌지주스 => 소주	 0.2	 1.0	 1.67	
콜라 => 소주, 와인	 0.2	 0.25	 1.25	
소주, 와인 => 콜라	 0.2	 1.0	 1.25	


In [31]:
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs', 'rhs', 'support', 'confidence', 'lift'])
# rules_df.loc[0] = ['와인', '오렌지', 0.4, 1.0, 1.25] 식으로 추가
idx= 0
for rule in rules:
    supprot = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([ data for data in item[0]])
        rhs = ', '.join([ data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx] = [lhs, rhs, supprot, 
                                 round(confidence, 2),
                                 round(lift,2)]
            idx += 1
            
rules_df = rules_df.sort_values(by=['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)            
rules_df

,lhs,rhs,support,confidence,lift
0,"콜라, 오렌지주스",소주,0.2,1.00,1.67
1,소주,"오렌지주스, 콜라",0.2,0.33,1.67
2,소주,콜라,0.6,1.00,1.25
3,맥주,콜라,0.4,1.00,1.25
4,"맥주, 소주",콜라,0.2,1.00,1.25
5,"맥주, 와인",콜라,0.2,1.00,1.25
6,"오렌지주스, 소주",콜라,0.2,1.00,1.25
7,"소주, 와인",콜라,0.2,1.00,1.25
8,콜라,소주,0.6,0.75,1.25
9,콜라,맥주,0.4,0.50,1.25


In [39]:
df = pd.read_csv('data/전주여행_경주여행_사람인.csv', sep='\t')
# df.drop(columns='Unnamed: 0', inplace=True)
total_text_list = df['total_text'].to_list()
total_text_list[0]

'전주 가볼만한곳 추천 받아요  추억의 7080 다양한체험 7080감성 추억여행 테마박물관 유익한시간 2 전북 전북투어패스 통합이용권 전북핫플 여러여행지 다양한체험 카페이용추가 전주여행필수 편안하고 즐거운 날이 되시길 바라겠습니다 감사합니다 '

In [47]:
%%time
from konlpy.tag import Hannanum, Kkma
#analyzer = Hannanum()
analyzer = Kkma()
total_noun_list = []
select_pos = ['NC', 'NQ']
select_pos = ['NNP', 'NNG']
불용어 = {'여행'}
for total_text in total_text_list:
    total_noun = [token for token, tag in analyzer.pos(total_text) \
                  if tag in select_pos and len(token) > 1 ]
    total_noun_list.append(total_noun)

total_noun_list

CPU times: total: 2min 4s
Wall time: 1min 56s


[['전주',
  '추천',
  '추억',
  '다양',
  '체험',
  '추억',
  '여행',
  '테마',
  '박물관',
  '유익',
  '시간',
  '투어',
  '패스',
  '통합',
  '이용권',
  '여행지',
  '다양',
  '체험',
  '카페',
  '이용',
  '추가',
  '전주',
  '여행',
  '필수',
  '편안',
  '감사'],
 ['전주',
  '여행',
  '전주',
  '여행',
  '사람',
  '호텔',
  '가격',
  '여행',
  '얼마',
  '정도',
  '카페',
  '추천',
  '주세',
  '안녕',
  '하세',
  '전주',
  '여행',
  '계획',
  '중이',
  '한옥',
  '마을',
  '근처'],
 ['전주',
  '여행',
  '관련',
  '질문',
  '전주',
  '여행',
  '계획',
  '선택',
  '전주',
  '한옥',
  '마을',
  '자연',
  '경관',
  '음식',
  '동네',
  '시간',
  '한옥',
  '마을',
  '근처',
  '장소'],
 ['학생',
  '친구',
  '전주',
  '여행',
  '안녕',
  '하세',
  '여학생',
  '친구',
  '전주',
  '여행',
  '가기',
  '일본',
  '전주',
  '오타쿠'],
 ['모닝',
  '전주',
  '여행',
  '부모님',
  '전주',
  '여행',
  '명소',
  '식당',
  '추천',
  '전주',
  '한옥',
  '마을',
  '유명',
  '완산구',
  '덕진구',
  '추억',
  '장소'],
 ['전주',
  '여행',
  '이번',
  '여자',
  '친구',
  '전주',
  '여행',
  '여행',
  '경로',
  '주세',
  '토요일',
  '실내',
  '데이트',
  '가능',
  '감사',
  '숙소',
  '전주',
  '한옥',
  '마을',
  '근처',
  '한옥',
  '마을',
  '구경'],
 ['전주',


In [59]:
%%time
from apyori import apriori
rules = apriori(total_noun_list, # 2차원 데이터
               min_support = 0.15,
               min_confidence=0.2)
rules = list(rules)
len(rules)

CPU times: total: 234 ms
Wall time: 226 ms


406

In [60]:
%%time
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs', 'rhs', 'support', 'confidence', 'lift'])
# rules_df.loc[0] = ['와인', '오렌지', 0.4, 1.0, 1.25] 식으로 추가
idx= 0
for rule in rules:
    supprot = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([ data for data in item[0]])
        rhs = ', '.join([ data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx] = [lhs, rhs, supprot, 
                                 round(confidence, 2),
                                 round(lift,2)]
            idx += 1
            
rules_df = rules_df.sort_values(by=['lift', 'confidence', 'support'], ascending=False).reset_index(drop=True)            
rules_df

CPU times: total: 13.5 s
Wall time: 13.5 s


,lhs,rhs,support,confidence,lift
0,"주가, 국내","볼만, 여수",0.157,1.00,5.99
1,"볼만, 국내","코스, 여수",0.157,1.00,5.99
2,"볼만, 국내","티비, 여수",0.157,1.00,5.99
3,"주가, 국내","코스, 여수",0.157,1.00,5.99
4,"주가, 국내","티비, 여수",0.157,1.00,5.99
...,...,...,...,...,...
9829,추천,"경주, 여행",0.217,0.51,1.01
9830,"추천, 여행",경주,0.217,0.51,1.01
9831,경주,추천,0.217,0.43,1.01
9832,경주,"추천, 여행",0.217,0.43,1.01


In [50]:
rules_df.head(20)

,lhs,rhs,support,confidence,lift
0,"주가, 국내","볼만, 여수",0.157,1.0,5.99
1,"볼만, 국내","코스, 여수",0.157,1.0,5.99
2,"볼만, 국내","티비, 여수",0.157,1.0,5.99
3,"주가, 국내","코스, 여수",0.157,1.0,5.99
4,"주가, 국내","티비, 여수",0.157,1.0,5.99
5,"주가, 국내","볼만, 여행, 여수",0.157,1.0,5.99
6,"주가, 여행, 국내","볼만, 여수",0.157,1.0,5.99
7,"볼만, 국내","여행, 여수, 코스",0.157,1.0,5.99
8,"볼만, 여행, 국내","여수, 코스",0.157,1.0,5.99
9,"볼만, 국내","여행, 티비, 여수",0.157,1.0,5.99
